In [ ]:
!pip install -U diffusers transformers accelerate huggingface_hub peft

In [ ]:
import os
os._exit(00)

In [ ]:
import torch
from diffusers import FluxPipeline
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# 1. Login
user_secrets = UserSecretsClient()
login(token=user_secrets.get_secret("HF_TOKEN"))

# 2. Load Model (Using Kaggle's native PyTorch)
pipe = FluxPipeline.from_pretrained(
    "kpsss34/FHDR_Uncensored", 
    torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()

print("✅ Model loaded successfully on clean environment.")

# 3. Generate
prompt = "luxury editorial photography of a high-fashion model, 85mm lens, sharp focus, cinematic lighting"
image = pipe(prompt, guidance_scale=3.5, num_inference_steps=28).images[0]
image.show()

In [ ]:
!pip install -U bitsandbytes

In [ ]:
import os
os._exit(00)

In [ ]:
import torch
from diffusers import FluxPipeline, FluxTransformer2DModel
from diffusers import BitsAndBytesConfig as DiffusersBitsAndBytesConfig
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# 1. Login
user_secrets = UserSecretsClient()
login(token=user_secrets.get_secret("HF_TOKEN"))

# 2. Setup the Q4 / 4-bit Configuration
quant_config = DiffusersBitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4" # Standard 4-bit format
)

print("Loading the heavy transformer in 4-bit (this saves ~16GB of VRAM)...")
# 3. Load ONLY the heavy transformer in 4-bit
transformer_4bit = FluxTransformer2DModel.from_pretrained(
    "kpsss34/FHDR_Uncensored",
    subfolder="transformer",
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16
)

print("Loading the rest of the pipeline...")
# 4. Load the full pipeline, plugging in our 4-bit compressed transformer
pipe = FluxPipeline.from_pretrained(
    "kpsss34/FHDR_Uncensored",
    transformer=transformer_4bit,
    torch_dtype=torch.bfloat16
)

# You can use standard model offload now instead of the slow sequential one!
pipe.enable_model_cpu_offload() 

# 5. Generate your image!
prompt = "Luxury editorial photography of a high-fashion model, 85mm lens, sharp focus, cinematic studio lighting"

image = pipe(
    prompt, 
    guidance_scale=3.5, 
    num_inference_steps=28,
    width=1024,
    height=1024
).images[0]

image.save("Q4_Fashion_Output.png")
from IPython.display import display

# WORKING IN KAGGLE
display(image)